# MVP - Machine Learning & Analytics PUC RJ

**Nome:** Luis Carlos Firmino Pinheiro  
**Matrícula:** 4052026000838  
**Data:** 13/06/2026  
**Dataset:** [SINISTROS DE TRÂNSITO - PRF](https://dados.gov.br/dados/conjuntos-dados/sinistros-de-transito-agrupados-por-ocorrencia)  
**Tipo de problema:** Classificação  

# 1. Definição do problema
## 1.1 Descrição do problema

No ano de 2025 a PRF registrou 72.529 sinistros de trânsito, com 83.550 feridos e 6.043 fatalidades.

Existem diversos estudos analisando informações históricas sobre ocorrências, quando foram, como ocorreram, o que as causaram. Mas e se utilizarmos esses mesmos dados em tempo real, durante a informação de uma ocorrência, seja por ligação ou por alertas gerados por aplicativos de navegação,  poderíamos com essas informações, saber se uma ocorrência será com ou sem feridos, ou até mesmo vitimas fatais, mesmo antes de se chegar ao local?

Será que um modelo preditivo tem um desempenho melhor que apenas uma analise exploratória dos locais mais frequentes, analise de risco e afins?  

Machine Learning nos permite analisar múltiplas interações e fatores ao mesmo tempo, e é dai que esperamos que o desempenho seja melhor do que uma analise exploratória. 

A ideia é que o modelo seja utilizado na linha de frente de centrais de atendimento a ocorrência para auxiliar na tomada de decisões, otimizando o tipo de resposta, equipes e recursos utilizados, com um foco especial em evitar falsos negativos de ocorrências graves, já que estamos lidando com vidas.

A base de dados que iremos utilizar é [SINISTROS DE TRÂNSITO - PRF](https://dados.gov.br/dados/conjuntos-dados/sinistros-de-transito-agrupados-por-ocorrencia) da PRF, que conta com dados de acidentes em rodovias brasileiras, categorizados por gravidade, causa, tipos, dentre outras informações relevantes ao acidente.

## 1.2 Objetivo do MVP

**Objetivo deste trabalho:**  
> O Objetivo deste MVP é construir e avaliar modelos de Machine Learning para prever a gravidade de um acidente a partir de informações disponíveis no momento em que ocorreu, comparando uma abordagem baseline com modelos candidatos e discutindo suas limitações.


## 1.3 Tipo de problema

**Tipo escolhido:** Classificação Multiclasse  
**Justificativa:** Estamos prevendo o rótulo de gravidade associado à ocorrência no campo: `classificacao_acidente` 
1. Sem Vítimas
1. Com Vítimas Feridas
1. Com Vítimas Fatais

## 1.4 Premissas, hipóteses e critérios de sucesso

**Hipóteses iniciais:**
1. Features de contexto como horário, localização, tipo de via, condição meteorológica e outros, carregam um sinal preditivo suficiente para superar o baseline, mesmo sem a descrição do que ocorreu no acidente. 

1. A analise de múltiplas interações entre variáveis que um modelo de machine learning desempenhará, ira resultar em um melhor desempenho do que o baseline.

1. O modelo tende a cometer menos erros de falso negativo na classe `Com vítimas fatais` do que o baseline.

1. O modelo tem um balanço melhor entre precisão e recall do que o baseline.

**Critérios de sucesso:**

- **Métrica principal:** **F1-score Macro** (Lida com classes desbalanceadas).

- **Métrica secundária:** Recall da classe mais crítica `Com Vítimas Fatais`.

- **Resultado mínimo esperado:** O Baseline simulará um sistema que tem acesso a informações históricas de frequências e distribuições estatísticas, e para isso será utilizado um `DummyClassifier` com `strategy="stratified"`, assim ele chutará baseando-se em probabilidades conhecidas, e não somente a classe majoritária. 

    Então o critério será de obter um **F1-Macro superior ao baseline em no mínimo 5 pontos percentuais**, com um **recall mínimo de 60% para a classe `Com Vítimas Fatais`**, garantindo que mais da metade dos acidentes gravíssimos sejam detectados preventivamente.

- **Por que 5 pontos percentuais no F1-Macro?** Por definição esse classificador terá desempenho no F1-Macro de aproximadamente 33,3% (1/n classes). Esses 5 pontos são para reduzir o risco de ruído estatístico, mas ainda com uma meta realista levando em consideração o recall exigido da classe fatal.    

- **Porque 60% de recall?:** Levando em consideração que a classe de acidentes fatais deve ser a minoritária no dataset o baseline irá classifica-la próximo de 6% a 8% ignorando por volta de 92 de 100 acidentes graves. A ideia é mudar esse paradigma para que sejam identificados a **maioria** dos acidentes graves.

- **Porque não 51% para considerar como maioria?:** Com 60% ainda teremos uma margem de segurança para variações aleatórias no modelo e o eventual drift ao longo do tempo.

- **Restrição prática:** As features usadas para inferência precisam estar disponíveis no momento da detecção/informe da ocorrência e antes da chegada da equipe ao local. 


# 2. Ambiente, bibliotecas e reprodutibilidade

### 2.1 Dependências adicionais

In [ ]:
!pip install -q lightgbm category_encoders holidays

### 2.2 Importações

In [ ]:
import random
import sys
import time
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import randint,uniform, loguniform

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.tree import DecisionTreeClassifier
from lightgbm import LGBMClassifier
from category_encoders import CatBoostEncoder
from category_encoders.wrapper import PolynomialWrapper


from sklearn.metrics import (
    accuracy_score,
    f1_score,
    fbeta_score,
    precision_score,
    recall_score,
    ConfusionMatrixDisplay,
    classification_report,
    make_scorer
)

import holidays

### 2.3 Configurações e ambiente

In [ ]:
pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)
warnings.filterwarnings(
    "ignore", category=UserWarning, message="X does not have valid feature names"
)
plt.style.use("ggplot")

SEED = 202606
np.random.seed(SEED)
random.seed(SEED)

print("Python:", sys.version.split()[0])
print("Seed:", SEED)

## 2.4 Funções e classes auxiliares

### Classes customizadas necessárias para pre-processamento e encoding

In [ ]:
class TracadoViaEncoder(BaseEstimator, TransformerMixin):
    """
    Faz one-hot encode da feature tracado_via, fazendo o tratamento do separador
    e transformando as caracteristicas de via em novas colunas booleanas (1, 0)
    """

    def __init__(
        self,
        sep=";",
        classes=[
            [
                "reta",
                "curva",
                "declive",
                "interseção de vias",
                "rotatória",
                "aclive",
                "viaduto",
                "retorno regulamentado",
                "em obras",
                "ponte",
                "desvio temporário",
                "túnel",
            ]
        ],
    ):
        self.sep = sep
        self.classes = classes

    def _parse(self, X):
        col = X.iloc[:, 0] if isinstance(X, pd.DataFrame) else pd.Series(np.ravel(X))
        return (
            col.astype(str)
            .str.lower()
            .str.split(self.sep)
            .apply(lambda lst: [v.strip() for v in lst if v.strip()])
        )

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        listas = self._parse(X)
        colunas = {
            f"tracado_{c}": listas.apply(lambda l: int(c in l)) for c in self.classes
        }
        return pd.DataFrame(colunas, index=listas.index).values

    def get_feature_names_out(self, input_features=None):
        return [f"tracado_{c}" for c in self.classes]


class ResolucaoKmTransformer(BaseEstimator, TransformerMixin):
    """
    Reduz latitude/longitude à resolução de ~1km, simulando coordenadas
    vindas de um sistema de lookup em vez de registro exato por GPS.
    """

    KM_POR_GRAU_LAT = 111.0

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_arr = np.asarray(X, dtype=float)
        lat, lon = X_arr[:, 0], X_arr[:, 1]

        km_por_grau_lon = self.KM_POR_GRAU_LAT * np.cos(np.radians(lat))

        lat_km = np.round(lat * self.KM_POR_GRAU_LAT) / self.KM_POR_GRAU_LAT
        lon_km = np.round(lon * km_por_grau_lon) / km_por_grau_lon

        return np.column_stack([lat_km, lon_km])

    def get_feature_names_out(self, input_features=None):
        return ["latitude_km", "longitude_km"]

def transform_ciclica(X, periodo):
    """
    Transforma variaveis numericas temporais em seu seno e cosseno.
    """
    X_arr = np.asarray(X, dtype=float).ravel()
    sen = np.sin(2 * np.pi * X_arr / periodo)
    cos = np.cos(2 * np.pi * X_arr / periodo)
    return np.column_stack([sen, cos])


def nomes_ciclica(nome_base):
    def _fn(transformer, input_features):
        return [f"{input_features[0]}_sen", f"{input_features[0]}_cos"]

    return _fn        

### Funções auxiliares

In [ ]:
def adicionar_feriados(df, coluna_data, coluna_uf, pais="BR"):
    """
    Adiciona flags (1 ou 0) para feriados, vespera e pos feriado
    considerando feriados estaduais com base em uma coluna de UF do dataframe
    """

    df[coluna_data] = pd.to_datetime(df[coluna_data])
    datas_limpas = df[coluna_data].dt.normalize()

    df["feriado"] = 0
    df["vespera_feriado"] = 0
    df["pos_feriado"] = 0

    for uf, grupo in df.groupby(coluna_uf):
        if pd.isna(uf):
            continue

        anos_presentes = grupo[coluna_data].dt.year.dropna().unique().tolist()

        try:
            feriados_obj = holidays.country_holidays(
                pais, subdiv=str(uf).upper(), years=anos_presentes
            )
        except (KeyError, ValueError):
            feriados_obj = holidays.country_holidays(pais, years=anos_presentes)

        datas_feriados = pd.to_datetime(list(feriados_obj.keys()))

        indices_grupo = grupo.index
        datas_grupo = datas_limpas.loc[indices_grupo]

        df.loc[indices_grupo, "feriado"] = datas_grupo.isin(datas_feriados).astype(
            "int8"
        )
        df.loc[indices_grupo, "vespera_feriado"] = (
            (datas_grupo + pd.Timedelta(days=1)).isin(datas_feriados).astype("int8")
        )
        df.loc[indices_grupo, "pos_feriado"] = (
            (datas_grupo - pd.Timedelta(days=1)).isin(datas_feriados).astype("int8")
        )

    return df


def reclassificar_ocorrencia(row):
    """
    Recupera a classificacao da ocorrencia baseando-se em outras colunas de dados
    """
    if row["mortos"] > 0:
        return "Com vítimas fatais"
    elif row["feridos"] > 0:
        return "Com vítimas feridas"
    else:
        return "Sem vítimas"


def tabela_percentual(df: pd.DataFrame, var: str, cat: str):
    """
    Cria um dataframe com os percentuais de variavel por categoria
    """
    data = df.groupby([var, cat]).size().unstack()
    data = data.div(data.sum(axis=1), axis=0) * 100

    return data


def plotar_grafico(
    data: pd.DataFrame | pd.Series,
    kind: str,
    title: str,
    x_label: str,
    y_label: str,
    figsize=(14, 6),
):
    """
    Plota um grafico generico a depender de alguns tipos pre configurados.
    """
    fig, ax = plt.subplots(figsize=figsize)

    data.plot(kind=kind, ax=ax)

    ax.set_title(title, loc="left")
    ax.set_xlabel(x_label, loc="left")
    ax.set_ylabel(y_label, loc="top")

    if kind == "bar":
        ax.get_yaxis().set_visible(False)
        ax.grid(visible=False)
        plt.xticks(rotation=0)

    elif kind == "barh":
        ax.get_xaxis().set_visible(False)

    if kind in ["bar", "barh"]:
        for container in ax.containers:
            ax.bar_label(container=container)

    plt.show()


def barras_empilhadas(
    data: pd.DataFrame | pd.Series,
    kind: str,
    title: str,
    x_label: str,
    y_label: str,
    category: str = "Categorias",
    figsize=(14, 6),
):
    """
    Plota um grafico de barras empilhadas (percentais de classe)
    """
    fig, ax = plt.subplots(figsize=figsize)

    data.plot(kind=kind, stacked=True, ax=ax, width=0.7)

    ax.set_title(title)
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)

    if kind == "bar":
        ax.tick_params(axis="x", rotation=0)
        ax.get_yaxis().set_visible(False)

    ax.grid(visible=False)

    for container in ax.containers:
        ax.bar_label(
            container=container,
            fmt="%.1f%%",
            label_type="center",
        )

    plt.legend(title=category, bbox_to_anchor=(1.01, 1), loc="upper left")

    plt.tight_layout()

    plt.show()


def evaluate_classification(
    y_true, y_pred, proba=None, classe_fatal="Com vítimas fatais"
):
    """
    Metricas de avaliacao para o problema de classificacao de gravidade
    (3 classes). F1-macro e a metrica principal, recall da classe fatal
    e usada como metrica secundaria.
    """

    results = {
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        f"recall_{classe_fatal}": recall_score(
            y_true, y_pred, labels=[classe_fatal], average="micro", zero_division=0
        ),
        "f2_macro": fbeta_score(
            y_true, y_pred, beta=2, average="macro", zero_division=0
        ),
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "precision_macro": precision_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        f"f2_{classe_fatal}": fbeta_score(
            y_true,
            y_pred,
            beta=2,
            labels=[classe_fatal],
            average="micro",
            zero_division=0,
        ),
    }
    return results


def show_results_table(results_dict):
    """Exibe resultados como ordenado por F1-macro."""
    df_results = pd.DataFrame(results_dict).T
    return df_results.sort_values(
        ["f1_macro", "recall_Com vítimas fatais"], ascending=False
    )


# 3. Seleção e carga dos dados

## 3.1 Fonte dos dados

Base de dados de sinistros em rodovias brasileiras agrupados por ocorrência.

- **Nome do dataset:** SINISTROS DE TRÂNSITO - PRF
- **Link da fonte:** https://dados.gov.br/dados/conjuntos-dados/sinistros-de-transito-agrupados-por-ocorrencia
- **Link no Github:** #TODO
- **Por que esse dataset foi escolhido?:** Volume expressivo de dados bem categorizados e com boa informação, aliados a um problema do mundo real e não dados sintéticos.
- **Quais restrições ou condições foram consideradas?:** O dataset possui dados registrados pós ocorrência, a restrição intencional é de usar apenas dados que seriam possíveis de ser obtidos antes da chegada de uma equipe ao local.
- **Se há questões de ética, privacidade, confidencialidade ou licença**: O dataset é de domínio público e não possui informações pessoais ou identificatórias.


## 3.2 Carga dos dados


In [ ]:
dataset_path = "./data/dados_prf.parquet"

df = pd.read_parquet(dataset_path)

df.head()

## 3.3 Visão geral do dataset e tratamento de dados

### Linhas e Colunas

In [ ]:
print(f"Linhas: {df.shape[0]}\nColunas: {df.shape[1]}")

### Tipos dos dados

In [ ]:
display(df.dtypes.to_frame("Tipo"))
print(f"Uso de memória: {df.memory_usage().sum() / 1e6:.1f} mb")

Temos oportunidades de reduzir o uso de memoria otimizando os tipos desses dados.  
Vamos normalizar os textos e tratar como categorias, além de reduzir a quantidade de bits de algumas variáveis numéricas.

In [ ]:
df["id"] = df["id"].astype(int)
df["data_inversa"] = pd.to_datetime(df["data_inversa"], format="mixed")
df["dia_semana"] = (
    df["dia_semana"]
    .str.lower()
    .str.strip()
    .str.removesuffix("-feira")
    .astype("category")
)
df["horario"] = pd.to_timedelta(df["horario"])
df["uf"] = df["uf"].str.upper().str.strip().astype("category")
df["br"] = df["br"].astype("Int32")  # Nan presente por isso não foi reduzido a int16
df["km"] = df["km"].str.replace(",", ".").astype("float32")  # Nan presente por isso não foi reduzido a int16
df["municipio"] = df["municipio"].str.upper().str.strip().astype("category")
df["causa_acidente"] = (
    df["causa_acidente"].str.strip().str.capitalize().astype("category")
)
df["tipo_acidente"] = (
    df["tipo_acidente"].str.strip().str.capitalize().astype("category")
)
df["classificacao_acidente"] = (
    df["classificacao_acidente"].str.strip().str.capitalize().astype("category")
)
df["fase_dia"] = df["fase_dia"].str.strip().str.capitalize().astype("category")
df["sentido_via"] = df["sentido_via"].str.strip().str.capitalize().astype("category")
df["condicao_metereologica"] = (
    df["condicao_metereologica"].str.strip().str.capitalize().astype("category")
)
df["tipo_pista"] = df["tipo_pista"].str.strip().str.capitalize().astype("category")
df["tracado_via"] = df["tracado_via"].str.strip().str.capitalize().astype("category")
df["uso_solo"] = df["uso_solo"].str.strip().str.capitalize().astype("category")
df["ano"] = df["ano"].astype("Int16")
df["pessoas"] = df["pessoas"].astype("int8")
df["mortos"] = df["mortos"].astype("int8")
df["feridos_leves"] = df["feridos_leves"].astype("int8")
df["feridos_graves"] = df["feridos_graves"].astype("int8")
df["ilesos"] = df["ilesos"].astype("int8")
df["ignorados"] = df["ignorados"].astype("int8")
df["feridos"] = df["feridos"].astype("int8")
df["veiculos"] = df["veiculos"].astype("int8")
df["latitude"] = df["latitude"].str.replace(",", ".").astype("float32")
df["longitude"] = df["longitude"].str.replace(",", ".").astype("float32")
df["regional"] = df["regional"].str.strip().str.upper().astype("category")
df["delegacia"] = df["delegacia"].str.strip().str.upper().astype("category")
df["uop"] = df["uop"].str.strip().str.upper().astype("category")


### Novos tipos do Dataset

In [ ]:
display(df.dtypes.to_frame("Tipo"))
print(f"Uso de memória: {df.memory_usage().sum() / 1e6:.1f} mb")

### Valores ausentes por coluna

In [ ]:
print("Valores ausentes por coluna")
df.isna().sum().to_frame("Ausentes")

**Temos valores ausentes nas colunas:**

- **`uf`, `br`, `km`:** Essas colunas são indicativos de onde ocorreu o acidente, sem isso não temos para onde enviar o socorro. Como são apenas 12 exemplos em um dataset de 2218342, não perderíamos muita informação ao remover essas linhas.

- **`causa_acidente`:**  É determinada após investigação do fato, então não será parte do modelo

- **`tipo_acidente`:**  Pode até ser conhecido no momento que a ocorrência é informada, mas vale analisar posteriormente se será mantida ou não, dependendo de quão específica é essa informação.

- **`classificacao_acidente`:** É o nosso alvo de predição, linhas sem esse alvo não tem valor para nós. Mas ainda vale a pena verificar se os casos sem essa classificacao foram apenas erro de preenchimento e se podemos derivar essa informação a partir do numero de vitimas feridas e mortos.

- **`fase_dia`:** É redundante pois `horario` já contém essa informação.

- **`condicao_metereologica`:** Pode ser um fator importante, mas a quantidade de registros em ausentes é mínima, não perderíamos muita informação ao eliminar essas linhas.

- **`tipo_pista`, `tracado_via`, `uso_solo`:** Também com poucos valores ausentes se comparado ao tamanho do dataset.

- **`ano`:** Tem muitos valores ausentes, mas pode ser recuperado extraindo essa informação da coluna `data_inversa`.

- **`latitude`, `longitude`:** Essas duas tem um volume expressivo de nulos, podemos verificar se foi uma informação que começou a ser preenchida a partir de um determinado ano, o que indicaria que houveram mudanças no processo de preenchimento. 

- **`regional`,`delegacia`, `uop`** Essas também tem um número expressivo de valores ausentes, semelhante ao de `latitude` e `longitude`, o que reforça a hipótese de que houveram mudanças de processo. Mas no caso dessas colunas elas são colunas processuais do registro de ocorrência, que só seriam conhecidas após ao fato, por premissa, ficam de fora do modelo.


### Duplicatas

In [ ]:
print("Duplicatas:", df.duplicated().sum())

### Enriquecimento de informações do dataset.

Uma das informações que acredito ser útil é a de feriados. Os feriados afetam a demanda e tráfego nas rodovias, então por intuição devem ser um influenciador para o modelo.
Além de feriados, vamos adicionar também vésperas e pós feriados como flags binárias.

Os feriados estaduais também serão considerados para as suas respectivas unidades federativas.

In [ ]:
df = adicionar_feriados(df, "data_inversa", "uf")

### Inspeção da variável alvo

In [ ]:
print("Valores da variável alvo")
df["classificacao_acidente"].value_counts(dropna=False).to_frame("n").reset_index()

Vamos inspecionar alguns desses NaN

In [ ]:
df[df["classificacao_acidente"].isna()].sample(15, random_state=SEED)

Há registros onde mesmo com o rótulo ausente, ainda temos informações nas colunas `mortos` e `feridos`. Podemos recuperar esses rótulos dessas linhas aplicando regras com base nesses valores. Podemos fazer isso tanto para os casos *NaN* quanto para os *Ignorado*.

Meu receio é de classificar *Ignorado* como *Sem vítima* quando a informação não for confiável.

Por isso vamos inspecionar esses casos primeiro.

Os NaN são poucos então vamos mostra-los diretamente.

In [ ]:
df[df["classificacao_acidente"].isna()][
    [
        "pessoas",
        "mortos",
        "feridos_leves",
        "feridos_graves",
        "ilesos",
        "ignorados",
        "feridos",
    ]
]

In [ ]:
df[df["classificacao_acidente"].isna()][
    [
        "pessoas",
        "mortos",
        "feridos_leves",
        "feridos_graves",
        "ilesos",
        "ignorados",
        "feridos",
    ]
].sum()

> Realmente temos informações de feridos, então faz sentido utilizar a regra nesses casos.

Agora para os Ignorados

In [ ]:
df[df["classificacao_acidente"] == "Ignorado"].sample(n=15, random_state=SEED)

In [ ]:
df[df["classificacao_acidente"] == "Ignorado"][
    [
        "pessoas",
        "mortos",
        "feridos_leves",
        "feridos_graves",
        "ilesos",
        "ignorados",
        "feridos",
    ]
].sum()

>Os dados com rotulo Ignorado não tem dados sobre feridos ou mortos, o que ajudaria na reclassificação.

OS tipos de acidentes com rotulo Ignorado

In [ ]:
df[df["classificacao_acidente"] == "Ignorado"]["tipo_acidente"].value_counts()

> Temos alguns acidentes com potencial grave nesse meio como Capotamento e Atropelamento de pessoa

Em que ano estão concentrados os `Ignorados`?

In [ ]:
df[df["classificacao_acidente"] == "Ignorado"].groupby(df["data_inversa"].dt.year)[
    "id"
].count()

> Os dados com rótulo de `Ignorado` so vão até o ano de 2016. Agora recordando daqueles dados que suspeitava ser por mudança de processo como `latitude` e `longitude`, vamos ver como eles se comportam também.

In [ ]:
df[df["latitude"].isna()].groupby(df["data_inversa"].dt.year).id.count()

> Como suspeitava, eles tambem só começaram a ser preenchidos após 2016

### Ponderação sobre utilização de dados anteriores a 2017

Os registro com a `classificacao_acidente` = `Ignorado` não apresentam nenhum dado de feridos ou mortos, ou seja, realmente não temos certeza do que ocorreu.

Temos alguns exemplos de tipos de acidente que aparentam ser graves como Atropelamento de pessoa, Capotamento, Colisão transversal, que teriam grande probabilidade de deixar feridos. Realmente não houveram feridos nesses acidentes ou não foi registrado no processo?

Isso aliado ao fato de que tivemos a inclusão de novas variáveis em 2017, me leva a crer que houve realmente uma mudança de processo no preenchimento das informações.

Diante disso acredito que o melhor caminho seja restringir esse estudo para o ano de 2017 em diante, por alguns motivos: 

- A mudança no padrão de preenchimento
- Mudanças nas normas de segurança, obrigatoriedade de airbags em novos veículos e atualização da infraestrutura rodoviária em um período de dados tão grande. 

Mesmo com essa restrição, ainda teremos quase 8 anos e meio de dados para construção do modelo.


### Reclassificacao de ocorrencias sem preenchimento

Vamos reclassificar os que podem ser reclassificados com base nas regras de feridos e mortos

In [ ]:
df["classificacao_acidente"] = df["classificacao_acidente"].astype("str")
filtro_nulos = df["classificacao_acidente"].isna()
nova_classificacao = df[filtro_nulos].apply(reclassificar_ocorrencia, axis=1)
df.loc[filtro_nulos, "classificacao_acidente"] = nova_classificacao

df["classificacao_acidente"] = df["classificacao_acidente"].astype("category")

In [ ]:
df["classificacao_acidente"].value_counts(dropna=False)

### Limitar o dataset somente a partir de 2017

Conforme discutido anteriormente, o periodo utilizado desse dataset será somente a partir de 2017

In [ ]:
df = df[df["data_inversa"].dt.year > 2016]

Confirmando a nova distribuição dos dados alvo

In [ ]:
df["classificacao_acidente"] = (
    df["classificacao_acidente"].astype("str").astype("category")
)

df["classificacao_acidente"].value_counts(dropna=False)

Como a variavel `ano` apresenta valores *NaN*, vamos usar o campo `data_inversa` para extrair o ano da ocorrência

In [ ]:
df["ano"] = df["data_inversa"].dt.year

Uma outra variável que quero tratar é a `uso_solo`, segundo informações do dicionario de dados da prf, ela informa se a ocorrência foi em trecho urbano ou rural

In [ ]:
df.uso_solo.value_counts()

Aparentemente antigamente havia a informação de trecho Rural ou Urbano no dataset anterior a 2017, mas agora os dados aparentam ser entre Sim e Não. Faz sentido alterar essa campo para uma flag (1, 0).

In [ ]:
df["uso_solo"] = (df["uso_solo"] == "Sim").astype("int8")

In [ ]:
df.uso_solo.value_counts()

### Tamanho final do dataset

In [ ]:
print(f"Novo shape do dataset\n Linhas: {df.shape[0]}\n Colunas: {df.shape[1]}")

## 3.4 Dicionário de dados (Pré Análise Exploratória EDA)

Tabela com os atributos do dataset.

| Coluna | Tipo | Descrição | Será usada no modelo? | Observações |
|---|---|---|---|---|
| id | numérica | id da ocorrência | não | identificador único de linha, sem informação para o modelo |
| data_inversa | data | data do acidente | parcialmente | será usada para ordenação do dataset e split temporal e não como feature|
| dia_semana | categórica  | dia da semana do acidente | sim | sim para capturar tendências durante a semana  |
| horario | timedelta | horário do acidente | sim | sim pois horário pode influenciar na visibilidade, demanda..etc |
| uf | categórica | unidade federativa onde ocorreu o acidente | sim | sim pois é essencial para localização |
| br | categórica | rodovia onde ocorreu o acidente | sim | sim pois é essencial para localização |
| km | numérica | km na rodovia onde ocorreu o acidente | sim | sim pois é essencial para localização |
| municipio | categórica | município do acidente | não | uf, br, km já nos dão informação suficiente sobre localização |
| causa_acidente | categórica | causa da ocorrência | não | determinada pós investigação, não é disponível ao modelo|
| tipo_acidente | categórica | tipo de acidente | não | é discutível se esse tipo de informação sempre estaria disponível ao modelo, portanto em primeiro momento será desconsiderada|
| fase_dia | categórica | fase do dia em que ocorreu | não | redundante com a coluna horario |
| sentido_via | categórica | sentido da via | sim | geralmente é possível determinar o sentido da via em algumas rodovias com mais facilidade, então vamos assumir que essa informação estará disponível |
| condicao_metereologica | categórica | condição meteorológica no local do acidente | sim | intuitivamente um forte influenciador |
| tipo_pista | categórica | tipo de pista onde ocorreu o acidente | *sim | *se temos a informação de uf, br e km, é razoável presumir que saberemos o tipo de pista, com base em algum lookup de banco de dados interno |
| tracado_via | categórica | traçado da pista | *sim | *se temos a informação de uf, br e km, é razoável presumir que saberemos o traçado de um determinado trecho, com base em algum lookup de banco de dados interno. Provavelmente será transformado para reduzir a cardinalidade |
| uso_solo | booleano | se foi em trecho urbano ou rual | sim | informação de característica da via que pode ser util, novamente assumimos que estaria disponível com lookup em base interna |
| ano | numérica | ano da ocorrência | sim | importante para informar o modelo sobre mudanças ao longo do tempo |
| pessoas | numérica | pessoas envolvidas | não | determinada pós o fato, não seria disponível ao modelo |
| mortos | numérica | mortos no acidente | não | determinada pós o fato, não seria disponível ao modelo, além de ser vazamento direto de alvo |
| feridos_leves | numérica | feridos levemente no acidente | não | determinada pós o fato, não seria disponível ao modelo, além de ser vazamento direto do alvo |
| feridos_graves | numérica | feridos gravemente no acidente | não | determinada pós o fato, não seria disponível ao modelo, além de ser vazamento direto do alvo |
| ilesos | numérica | ilesos no acidente | não | determinada pós o fato, não seria disponível ao modelo |
| ignorados | numérica | pessoas com estado desconhecido | não | determinada pós o fato, não seria disponível ao modelo |
| feridos | numérica | feridos no acidente | não | determinada pós o fato, não seria disponível ao modelo, além de ser vazamento direto do alvo |
| veiculos | numérica | veiculo envolvidos no acidente | não | discutível se essa informação estaria disponível ao modelo, portanto será descartada |
| latitude | numérica | latitude do acidente | indeterminado ! | ! aqui estou assumindo que latitude e longitude seria uma informação muito precisa para determinar no momento da ocorrência, mas pode se argumentar que poderíamos ter um lookup baseado em br + km, mas vamos explorar isso melhor adiante |
| longitude | numérica | longitude do acidente | indeterminado ! | ! aqui estou assumindo que latitude e longitude seria uma informação muito precisa para determinar no momento da ocorrência, mas pode se argumentar que poderíamos ter um lookup baseado em br + km, mas vamos explorar isso melhor adiante |
| regional | categórica | regional onde foi registrado o acidente | não | informação processual pós fato |
| delegacia | categórica | delegacia onde foi registrado o acidente | não | informação processual pós fato |
| uop | categórica | uop onde foi registrado o acidente | não | informação processual pós fato |
| feriado | booleano | se o dia foi feriado | sim | afeta demanda |
| vespera_feriado | booleano | se o dia foi véspera de feriado | sim | afeta demanda |
| pos_feriado | booleano | se o dia foi volta de feriado | sim | afeta demanda |
| **classificacao_acidente** | **alvo** | como o acidente foi classificado | alvo | Sem vítimas, Com vítimas feridas, Com vítimas fatais |

# 4. Análise exploratória dos dados


### Distribuição da variável alvo

In [ ]:
print("Distribuiçao do alvo:")

dist_alvo = (
    df["classificacao_acidente"]
    .value_counts(dropna=False)
    .to_frame("contagem")
    .reset_index()
)

dist_alvo["percentual"] = dist_alvo["contagem"] / dist_alvo["contagem"].sum() * 100

dist_alvo

> Temos um problema de classificação com classes desbalanceadas. A classe **Com vítimas feridas** aparece em quase 3/4 dos casos, seguido de **Sem vítimas** com quase 20%, e por fim **Com vítimas fatais** próximo de 7% dos casos. 

> Esse desbalanceamento pode fazer com que prever a classe majoritária seja um atalho para o modelo, por isso F1 Macro será usado como métrica para desempenho do modelo. O F1 Macro calcula o desempenho isoladamente para cada classe, utilizando a média harmônica entre precisão e sensibilidade.

### Tendência temporal

Mesmo que `data_inversa` tenha ficado de fora como feature, afinal a gravidade de um acidente que acontece hoje não tem influência na gravidade de um acidente amanhã, acredito que é importante manter a ordenação dos dados, já que o modelo poderia ter acesso a acidentes futuros durante o treinamento. Além do que a evolução natural do tempo, com melhorias (ou pioras) de infraestruturas podem ser capturadas por essa passagem de tempo.

Como proxy de passagem de tempo, utilizaremos o ano, assim o modelo entende que 2021 < 2022 e assim por diante.

In [ ]:
df = df.sort_values(by=["data_inversa", "horario"]).reset_index(drop=True)

Adicionando algumas variáveis para facilitar a exploração.  
**Não necessariamente se tornarão features**

In [ ]:
meses = {
    1: "Jan",
    2: "Fev",
    3: "Mar",
    4: "Abr",
    5: "Mai",
    6: "Jun",
    7: "Jul",
    8: "Ago",
    9: "Set",
    10: "Out",
    11: "Nov",
    12: "Dez",
}

dias = {
    0: "Seg",
    1: "Ter",
    2: "Qua",
    3: "Qui",
    4: "Sex",
    5: "Sáb",
    6: "Dom",
}

df["mes_ano"] = df["data_inversa"].dt.to_period("M")
df["mes"] = df["data_inversa"].dt.month
df["nome_mes"] = df["data_inversa"].dt.month.map(meses)
df["dia_semana"] = df["data_inversa"].dt.day_of_week
df["nome_dia"] = df["data_inversa"].dt.day_of_week.map(dias)
df["hora"] = df["horario"].dt.components["hours"]


### Como o número de acidentes mudou ao longo do tempo?

In [ ]:
data = df.groupby("mes_ano").size()

plotar_grafico(
    data,
    "line",
    "Evolução do número de acidentes ao longo do tempo",
    "Período",
    "Acidentes",
)

> Olhando esse gráfico podemos notar que houve uma redução no número geral em comparação ao início do período, mas agora aparenta retomar uma tendência de alta. 
>
> Também é possível notar que existe um componente de sazonalidade, com alguns periódos de redução no início do ano, e de alta no meio do ano para o final.
>
> Uma varíavel de mês pode ser importante para o modelo.

#### Vamos inspecionar melhor a tendência anual

In [ ]:
data = df.groupby("ano").size()

plotar_grafico(data, "bar", "Número de Acidentes por Ano", "Ano", "Acidentes")

> Claramente houve uma redução de 2017 para cá, mas temos que lembrar que 2020 a 2021 foram períodos de pandemia, oque pode ter influenciado e muito nessa redução. A tendência de alta nos anos recentes parece mais uma retomada a patamares pré pandemia.

> O ano de 2017 também se sobressai muito em comparação aos demais, oque foi diferente nesse ano?

> *Obs: 2026 ainda está incompleto*

Vamos verificar rapidamente os anos anteriores, para identificar se 2017 é mesmo um outlier

In [ ]:
pd.to_datetime(
    pd.read_parquet(dataset_path)["data_inversa"], format="mixed", dayfirst=True
).dt.year.value_counts().sort_index()

> Na verdade ja existia uma tendência de redução no número de acidentes, 2017 pode parecer destoante se olharmos apenas o período recente, mas na verdade ja tivemos anos muito piores.

Como o nosso objetivo é prever a gravidade, vamos ver como ela se comporta ao longo dos anos.

In [ ]:
data = (
    pd.crosstab(
        index=df["ano"], columns=df["classificacao_acidente"], normalize="index"
    )
    * 100
)


barras_empilhadas(
    data, "bar", "Classificações de acidentes por ano", "Ano", "Acidentes"
)


> Com exceção de 2017 e 2018 os pecentuais permanecem próximos ao longo dos anos

### Tendência Mensal

Vamos analisar a média de acidentes que ocorrem em um determinado mês

In [ ]:
data = df.groupby(["ano", "mes"]).size().groupby("mes").mean().round()

data = data.rename(index=meses)

plotar_grafico(data, "bar", "Média mensal de acidentes", "Mês", y_label="Acidentes")

> Como podemos ver em média Dezembro é o mês com maior número de acidentes, o que é esperado pelas festas de fim de ano. Mas será que a classificação também muda?

In [ ]:
data = (
    df.groupby(["ano", "mes", "classificacao_acidente"])
    .size()
    .groupby(["mes", "classificacao_acidente"])
    .mean()
    .unstack()
)

data = data.div(data.sum(axis=1), axis=0) * 100

data = data.rename(index=meses)

barras_empilhadas(data, "bar", "Classificação de acidentes por mês", "Mês", "%")

> Aparentemente a classificação por mês é bem estável mantendo uma proporção semelhante ao longo do ano, o que varia mesmo é o volume

> 

### Acidentes por dia da semana

In [ ]:
data = (
    df.groupby(["data_inversa", "dia_semana"])
    .size()
    .groupby("dia_semana")
    .mean()
    .round()
)

data = data.rename(index=dias)

plotar_grafico(
    data, "bar", "Média de Acidentes por dia da semana", "Dia da Semana", "Média"
)

> Como já é esperado, os dias com maior volume são aos finais de semana sex -> dom.
>
> Mas o dia afeta a classificação?

In [ ]:
data = (
    df.groupby(["data_inversa", "dia_semana", "classificacao_acidente"])
    .size()
    .groupby(["dia_semana", "classificacao_acidente"])
    .mean()
    .unstack()
)

data = data.rename(index=dias)

data = data.div(data.sum(axis=1), axis=0) * 100


barras_empilhadas(
    data,
    "bar",
    "Classificação de acidentes por dia da semana",
    "Dia da semana",
    "Média",
)


> Sim existe uma diferença nas gravidades, aos sábados e domingos a gravidade fatal aumenta, mas também aumentam os acidentes sem vítimas. 

### Acidentes por faixa horária

In [ ]:
data = df.groupby(["data_inversa", "hora"]).size().groupby("hora").mean().round()

plotar_grafico(data, "bar", "Média de acidentes por horario", "hora", "Média")


> Temos dois picos distintos nas médias de ocorrências, um pela manhã entre 7h e 8h e outro no final do dia as 18h, coincidente com os horários de pico de maior movimento das pessoas indo ao trabalho e retornando para casa.
> Mas suponho que nesses horários os acidentes sejam menos graves pela redução de velocidade das vias devido ao trânsito.

In [ ]:
data = (
    df.groupby(["data_inversa", "hora", "classificacao_acidente"])
    .size()
    .groupby(["hora", "classificacao_acidente"])
    .mean()
    .unstack()
)

data = (data.div(data.sum(axis=1), axis=0) * 100).sort_index(ascending=False)


barras_empilhadas(
    data,
    "barh",
    "Classificação de acidentes por faixa horaria",
    "%",
    "Horário",
    figsize=(10, 10),
)


> Embora as noites e madrugadas tenham menos acidentes em números absolutos, a proporção de acidentes graves é muito maior do que nos demais horários com maior volume.

### Sobre as variaveis temporais

Os dados demonstraram que é valido incluir essas variáveis por demonstrar diferenças entre padrões de comportamento nas classes alvos.

Só que acredito ser mais valido incluir essas variáveis como horário, dia da semana e mês, como "ciclos" ao invés de simplesmente utilizar o número delas. Afinal são variáveis que embora pareçam ordenadas, tem uma "virada" inerente ao seu tipo de dados, como por exemplo horário, 0h é menor que 23h mas 0h também vem depois de 23h.

Durante a etapa de pre-processamento vamos transformar essas variáveis em seus senos e cossenos, assim conseguimos representa-las como um "ciclo".

### Dados de localização  

#### UF - Unidade Federativa

In [ ]:
data = tabela_percentual(df, "uf", "classificacao_acidente")

barras_empilhadas(
    data, "barh", "Classificacao dos acidentes por UF", "%", "UF", figsize=(14, 10)
)


> Diferentes estados apresentam taxas diferentes de gravidade, Maranhão por exemplo é o com a pior taxa de fatalidade, Amazonas com a maior taxa de acidentes sem vitimas, entre outras particularidades por estado. Me parece uma informação útil ao modelo, além de trazer outras informações implícitas, como administração e infraestrutura.
>
> Podemos adotar o caminho de one-hot encode, mas isso criaria 26 novas variáveis (1 é descartada no get_dummies), ou podemos utilizar target-encoding que nos deixaria com 3 novas variáveis apenas. Só temos que tomar cuidado para aplicar o encoding levando em consideração a ordenação temporal para que a taxa de gravidades daquele estado seja aquela calculada somente até a linha atual.

### BR e KM

Chegamos a um ponto importante da analise. Embora sejam números, BR + KM informam uma localidade, como um espécie de endereço.

Temos dezenas de BR diferentes, não me parece sensato tranformar cada uma em uma nova coluna no dataset. Seria mais uma candidata a target encode.

Agora o Km é problematico, pois sem a informação do UF + BR, ele não quer dizer muita coisa sozinho. Isso me faz pensar se utilizar a Latitude e Longitude como substitutos do km não faria mais sentido.

In [ ]:
print("Número de BRs unicas no dataset:")

df["br"].nunique()

In [ ]:
data = tabela_percentual(df, "br", "classificacao_acidente")

data = data.sort_values(by="Com vítimas fatais", ascending=False).head(30)

barras_empilhadas(
    data,
    "barh",
    "Classificacao dos acidentes ordenados pelas 30 BRs com maior percentual de fatalidades",
    "%",
    "BR",
    figsize=(14, 12),
)

> O percentual por BR realmente traz informação útil, já que temos uma diferença notável entre elas. Os que mais saltam aos olhos são as brs 434 e 403 com mais de 50% de vitimas fatais. Um número até assustador, significa que se você sofrer um acidente em uma dessas BRs, a maior probabilidade é de que você seja uma vitima fatal. Algo que o percentual global de classes esconde, já que nele a classe fatal aparece com apenas 7%

Voltando ao problema do km

In [ ]:
df["km"].nunique()

> 9911 kms diferentes, lembrando que km embora seja um número, isolademente não traz informação útil, precisa do UF e BR para fazer sentido. Mas o número de combinações possíveis de UF + BR + Km iria explodir, o que pode deixar o modelo específico demais e com dificuldade em generalizar.

> Proponho voltarmos a utilizar as variáveis de latitude e longitude, mas vamos levar em conta que seria possível ter latitude e longitude aproximadas a partir da informação inicial de UF + BR + KM da ocorrência. A limitação como mencionei é de que não deve ser a latitude e longitudes exatas do acidente mas uma aproximação. 

> Como as coordenadas servirão como substitutos do km, proponho reduzir a resolução delas para 1 km no mínimo.

#### Como está a qualidde de latitude e longitude?

In [ ]:
df[["latitude", "longitude"]].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
)

> Temos alguns outliers, como latitudes de -1,033,382,848.00 e 163.00 e longitudes de -4,016,764,416.00 e 717.00.   
> Vamos fazer uma verificação mais específica baseada nos limites possíveis no território Brasileiro.

As coordenadas geográficas possíveis do Brasil são as seguintes:

| Referência | Latitude | Longitude |
|---|---|---|
| Extremo Norte (Monte Caburaí, RR) | +5,27° N | - |
| Extremo Sul (Arroio Chuí, RS) | -33,75° S | - |
| Extremo Oeste (fronteira AC/Peru) | - | -73,99° O |
| Extremo Leste (Ponta do Seixas, PB) | - | -34,79° O |
| **Range de validação (Arredondado para cima)** | **-34,0 a +5,5** | **-74,5 a -34,0** |

Vamos verificar quantos dados temos fora dessas coordenadas

In [ ]:
LAT_MIN, LAT_MAX = -34.0, 5.5
LON_MIN, LON_MAX = -74.5, -34.0


mask_fora_range = ~df["latitude"].between(LAT_MIN, LAT_MAX) | ~df["longitude"].between(
    LON_MIN, LON_MAX
)

df_geo_invalido = df[mask_fora_range]


print(f"Linhas invalidas: {df_geo_invalido.shape[0]}")

display(df_geo_invalido.groupby(["classificacao_acidente"]).size())


> 54 casos de latitudes e longitudes impossiveis. Um número desprezível se comparado ao tamanho total do dataset, acredito que podemos descartar esses dados com confiança.

In [ ]:
df = df[~mask_fora_range].reset_index(drop=True)

### Conclusão sobre latitude e longitude como substituto de km

Conforme discutido anteriormente, latitude e longitude pode ser um dado muito preciso para se saber no momento do acidente, mas podemos presumir que seria utilizado um sistema de busca de coordenadas com base em um uf + br + km, algo que é facilmente obtido até com ferramentas gratuitas.

Mas como nessa situação também não teriamos a latitude e longitude exata como é o registrado na ocorrência, vamos adicionar incerteza aos dados, reduzindo resolução da latitude e longitude para a escala de 1 quilômetro. Assim podemos considerar que esses dados seriam possíveis de obter antes da chegada da equipe ao local.

## Condição meteorológica

Como a condição metereologica influencia na gravidade?

In [ ]:
data = tabela_percentual(df, "condicao_metereologica", "classificacao_acidente")


barras_empilhadas(data, "barh", "Classes por condição metereologica", "%", "Condição")

> Claramente a condição do tempo tem influência na gravidade. `Neblina` por exemplo é onde a probabilidade de um acidente fatal é maior, e estranhamente dias de granizo são com a menor.
>
> Temos também a condição `Ignorado`. Vou optar por manter, para que sirva para simular por exemplo uma indisponibilidade de consulta a dados meteorológicos.


### Características da Via

Vamos ter uma visão geral sobre a influência das características da via

In [ ]:
data = tabela_percentual(df, "sentido_via", "classificacao_acidente")


barras_empilhadas(data, "barh", "Classes por Sentido da via", "%", "Sentido")

> Nessa variável não temos uma diferença muito grande entre `Decrescente` e `Crescente`, mas o `Não informado` apresenta uma distinção maior. Vou tomar a decisão de manter essa variável para que funcione como um "confundidor" do modelo, introduzindo um nível de incerteza realista, já que nem sempre teremos essa informação exata.

In [ ]:
data = tabela_percentual(df, "tipo_pista", "classificacao_acidente")


barras_empilhadas(data, "barh", "Classes por Tipo de Pista", "%", "Tipo")

> Pista Simples é um forte indicador de gravidade, e é esperado por serem pistas onde ocorrem ultrapassagens onde é necessário adentrar uma via com veículos em sentidos opostos.

Antes de plotar o traçado, vamos investigar a quantidade única dessa variável.

In [ ]:
df["tracado_via"].nunique()

> Uma combinação muito alta de possibilidades, 1369, vamos ver alguns exemplos

In [ ]:
df["tracado_via"].sample(20, random_state=SEED)

> Temos alguns separados por `;`, por isso a quantidade tão grande de dados únicos. Quais seriam as possibilidades únicas de características da via que podem ser preenchidas?

In [ ]:
df["tracado_via"].astype(str).str.split(";").explode().str.lower().str.strip().unique()

> Somente 12 possibilidades, bem mais razoável de se fazer one-hot-encode do que as 1369 anteriores iniciais.

In [ ]:
data = df.assign(
    tracado_via=df["tracado_via"].astype("str").str.lower().str.strip().str.split(";")
)

data = data.explode("tracado_via").reset_index(drop=True)

data = (
    pd.crosstab(
        data["tracado_via"],
        data["classificacao_acidente"],
        normalize="index",
    )
    * 100
)

barras_empilhadas(data, "barh", "Acidentes por Traçado de via", "%", "traçado")

> O traçado também carrega informação relevante, se acontecer em uma rotatória por exemplo a chance de vitimas fatais é bem menor, já se acontecer em uma ponte, a chance de fatalidade sobe.

In [ ]:
data = tabela_percentual(df, "uso_solo", "classificacao_acidente")


barras_empilhadas(data, "bar", "Classes por Trecho Urbano ou Rural", "%", "Tipo")

> Uso Solo também apresenta uma diferença notavel no comportamento das classes, vale a pena manter

### Feriados

Como os feriados influenciam na gravidade do acidente

In [ ]:
data = tabela_percentual(df, "feriado", "classificacao_acidente")

barras_empilhadas(data, "bar", "Gravidade em feriados (1) sim, (0) não", "Feriado", "%")

> Feriados apresentam ~= 0.6 pontos percentuais de diferença na classe fatal. Esperava que fosse maior mas ainda sim existe diferença.

In [ ]:
data = tabela_percentual(df, "vespera_feriado", "classificacao_acidente")

barras_empilhadas(
    data,
    "bar",
    "Gravidade em véspera de feriados (1) sim, (0) não",
    "Véspera-Feriado",
    "%",
)

>  A diferença em vésperas é bem menor, talvez até irrelevante

In [ ]:
data = tabela_percentual(df, "pos_feriado", "classificacao_acidente")

barras_empilhadas(
    data, "bar", "Gravidade em pós feriados (1) sim, (0) não", "Pós-Feriado", "%"
)

> Em pós feriado a classe fatal diminui, mas também muito pouco.
>
> Essas variáveis não foram tão relevantes quanto inicialmente se esperava. Mas talvez o seu poder explicativo esteja na combinação delas com outros dados como mês e horário.

### Check de nulos nas variáveis candidatas.

In [ ]:
df[[
    "dia_semana",
    "uf",
    "br",
    "sentido_via",
    "condicao_metereologica",
    "tipo_pista",
    "tracado_via",
    "uso_solo",
    "ano",
    "latitude",
    "longitude",
    "feriado",
    "vespera_feriado",
    "pos_feriado",
    "mes",
    "hora",
]].isna().sum()

> 0 valores ausentes

## 4.1 Síntese da análise exploratória

- Identificamos que estamos lidando com um problema de classes desbalanceadas, então realizar esse balanceamento durante o treino do modelo será essencial, além de utilizar uma métrica que leve em consideração esse desbalanceamento, que no caso será a métrica de **F1 Macro**


- A maioria das variáveis que inicialmente estavam como candidatas a features permaneceram, outras deverão ser transformadas, e outras foram descartadas:

- Variáveis temporais como `mes`, `dia da semana` e `horario` deverão ser transformadas em seu seno e cosseno para capturar o comportamento cíclico e sazonal.

- UF e BR serão transformadas em Target Encode.

- Km deixou de ser uma feature sendo substituída pela latitude e longitude, com a consideração que sua resolução será reduzida a fim de simular um ambiente de uso real.

- Condicao Metereologica, Sentido Via e Tipo Pista, Uso Solo serão one-hot-encode.

- Traçado da via também será one-hot mas com o tratamento da string para extrair todas as classes de dentro dela.

- feriado, vespera_feriado, pos_feriado permanecem, mas não tiveram tanta expressão na alteração das classes como pensei que teriam.


- Não temos valores ausentes, ou seja não será necessário imputação.

- Variáveis com diferença de escala: `ano`, `latitude`, `longitude`, `mes`, `dia_semana`, `hora`, mas essas 3 últimas terão transformação trigonométrica.


# 5. Preparação dos dados e divisão treino/teste

### Separação de features e target

In [ ]:
TARGET = "classificacao_acidente"
PROBLEM_TYPE = "classificacao"
DATE_COLUMN = "data_inversa"
ID_COLUMNS = ["id"]
DROP_COLUMNS = [
    "km",
    "municipio",
    "horario",
    "causa_acidente",
    "tipo_acidente",
    "fase_dia",
    "pessoas",
    "mortos",
    "feridos_leves",
    "feridos_graves",
    "ilesos",
    "ignorados",
    "feridos",
    "veiculos",
    "regional",
    "delegacia",
    "uop",
    "mes_ano",
    "nome_mes",
    "nome_dia",
    "data_inversa",
]

columns_to_exclude = set(ID_COLUMNS + DROP_COLUMNS)
columns_to_exclude.add(TARGET)

features = [c for c in df.columns if c not in columns_to_exclude]

print("Tipo de problema:", PROBLEM_TYPE)
print("Target:", TARGET)
print("Número de features:", len(features))
print("Features:", features)


### Divisão Treino e Teste

Embora esse seja um problema de classificação, a ordenação dos dados é importante para que o modelo não obtenha informação sobre acidentes do futuro para prever classes do passado. Além de que os padrões de infraestrutura e segurança mudaram ao longo do tempo, passamos por eventos adversos como a pandemia de 2020-2021, entre outros acontecimentos que podem influenciar no desempenho do modelo.

Vamos utilizar um holdout dos anos mais recentes (2025, 2026) como o nosso dataset de testes, e treinar com os anos anteriores.


In [ ]:
df_sorted = df.sort_values(DATE_COLUMN).copy()

data_corte = "2025-01-01"

train_df = df_sorted[df_sorted[DATE_COLUMN] < data_corte]
test_df = df_sorted[df_sorted[DATE_COLUMN] >= data_corte]

X_train, y_train = train_df[features].copy(), train_df[TARGET].copy()
X_test, y_test = test_df[features].copy(), test_df[TARGET].copy()

print(f"Treino: {X_train.shape} | Teste (Holdout): {X_test.shape}")


## 5.1 Justificativa da divisão

- **Por que usar holdout, validação cruzada ou divisão temporal?:**
    Utilizaremos uma divisão temporal, para que o target encoding não tenha acesso a taxas de gravidades posteriores ao da data da ocorrência. Além disso poderemos avaliar o desempenho e generalização do modelo com os dados mais recentes, e com um período maior do que de um ano, teremos como testar se o modelo reconhece a sazonalidade dos dados.


- **A proporção treino/teste faz sentido para o tamanho do dataset?:**
    Com essa divisão ficamos como uma proporção de 17% para teste e 83% para treino. Como demonstrado no EDA que as proporções entre as classes ao longo dos anos recentes permaneceu estável, ou seja o conjunto recente será representativo.

- **Foi necessário estratificar as classes?**: Estratificar não seria compatível com a ordenação temporal, mas será necessário utilizar balanceamento no treino dos modelos.

- **Como a divisão evita vazamento de dados?:**
    Com esse holdout de dados a partir de 2025, garantimos que o modelo não tenha acesso a informações de taxas de gravidades recentes durante o treinamento.

# 6. Pré-processamento e pipeline

### Preprocessamento

Tipos de dados

In [ ]:
target_enc_cols = ["uf", "br"]
ohe_cols = ["condicao_metereologica", "sentido_via", "tipo_pista"]
encoding_ciclico = ["mes", "dia_semana", "hora"]
tracado_col = ["tracado_via"]
geo_cols = ["latitude", "longitude"]
temporal_cols = ["ano"]
flag_cols = ["feriado", "vespera_feriado", "pos_feriado", "uso_solo"]

Pipeline

In [ ]:
# Transformacao e escalonamento das coordenadas
geo_pipe = Pipeline(
    [
        ("resolucao_km", ResolucaoKmTransformer()),
        ("scaler", StandardScaler()),
    ]
)


preprocess = ColumnTransformer(
    transformers=[
        # PolumialWrapper + CatBoostEncode sao compatives com a ordenacao dos dados,
        # por isso foi preferido ao target-encoder do sklearn
        (
            "target_enc",
            PolynomialWrapper(CatBoostEncoder(cols=target_enc_cols)),
            target_enc_cols,
        ),

        # one hot padrao do sklearn
        (
            "onehot_enc",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            ohe_cols,
        ),

        # Classe customizada para tratar os valores dentro da coluna de tracado_via
        ("tracado_enc", TracadoViaEncoder(), tracado_col),

        # Variaveis ciclicas com seno e cosseno
        (
            "ciclica_mes",
            FunctionTransformer(
                transform_ciclica,
                kw_args={"periodo": 12},
                feature_names_out=nomes_ciclica("mes"),
            ),
            ["mes"],
        ),
        (
            "ciclica_dia",
            FunctionTransformer(
                transform_ciclica,
                kw_args={"periodo": 7},
                feature_names_out=nomes_ciclica("dia_semana"),
            ),
            ["dia_semana"],
        ),
        (
            "ciclica_hora",
            FunctionTransformer(
                transform_ciclica,
                kw_args={"periodo": 24},
                feature_names_out=nomes_ciclica("hora"),
            ),
            ["hora"],
        ),
        ("geo", geo_pipe, geo_cols),
        ("ano_scaled", StandardScaler(), temporal_cols),
        ("flags", "passthrough", flag_cols),
    ],
    remainder="drop",
)


print("Target Encoding:", target_enc_cols)
print("One-Hot Encoding:", ohe_cols)
print("Multi-label (tracado_via):", tracado_col)
print("Encoding ciclico:", encoding_ciclico)
print("Geo (resolução 1km + StandardScaler):", geo_cols)
print("Ano (StandardScaler):", temporal_cols)
print("Passthrough (flags):", flag_cols)

preprocess


## 6.1 Decisões de pré-processamento

- Como não existem nulos nas features não será necessário imputação.

- `uf` e `br` passam por target encoding para capturar os fatores de risco das rodovias e estados. Nesse passo não é utilizado shuffle para que se mantenha a ordem temporal.

- `condicao_metereologica`, `sentido_via`, `tipo_pista` e `uso_solo` passam por one-hot encoding padrão.

- `tracado_via` passa por um tratamento para extrair as possibilidades de caracteristca da via e aplicado em um one-hot encode.

- `mes`, `dia_semana` e `hora` foram transformados em seno e cosseno para extrair comportamento ciclico e sazonal.

- `latitude` e `longitude` passam por redução de resolução simulando um sistema de busca por trecho e não as coordenadas exatas de gps.

- Realizei a padronização das variaveis numericas `ano`, `latitude` e `longitude`, para que sejam compativeis com modelos lineares (Regressão Logistica por exemplo)

- `km` foi removido por não ser comparável entre rodovias diferentes sem contexto adicional e foi substituido por `latitude` e `longitude`. `data_inversa` também foi removida, foi usada apenas para ordenação do dataset.

- Os demais campos removidos, se tratam ou de campos indisponiveis no momento do acidente, vazamentos diretos de classe ou irrelevantes.



# 7. Baseline e modelos candidatos

- O Baseline conforme descrito antes será um DummyClassifier com strategy "stratified"

- Testaremos uma gama de modelos de diferentes abordagens, Linear, Árvore, Bagging, Boosting e Voting

- Utilizaremos o parâmetro de class_weight="balanced" para levar em conta o desbalanceamento severo entre as classes.

In [ ]:
baseline = Pipeline(
    steps=[
        ("preprocess", preprocess),
        (
            "model",
            DummyClassifier(
                strategy="stratified",
                random_state=SEED,
            ),
        ),
    ]
)

candidates = {
    "LogisticRegression": Pipeline(
        steps=[
            ("preprocess", preprocess),
            (
                "model",
                LogisticRegression(
                    max_iter=500,
                    class_weight="balanced",
                    random_state=SEED,
                ),
            ),
        ]
    ),
    "DecisionTree": Pipeline(
        steps=[
            ("preprocess", preprocess),
            (
                "model",
                DecisionTreeClassifier(
                    max_depth=50,
                    class_weight="balanced",
                    random_state=SEED,
                ),
            ),
        ]
    ),
    "RandomForest": Pipeline(
        steps=[
            ("preprocess", preprocess),
            (
                "model",
                RandomForestClassifier(
                    class_weight="balanced",
                    random_state=SEED,
                    n_jobs=-1,  
                ),
            ),
        ]
    ),
    "LightGBM": Pipeline(
        steps=[
            ("preprocess", preprocess),
            (
                "model",
                LGBMClassifier(
                    objective="multiclass",
                    num_class=3,
                    class_weight="balanced",
                    random_state=SEED,
                    n_jobs=-1,  
                    verbosity=-1,
                ),
            ),
        ]
    ),
}

estimadores_voting = [
    ("LR", candidates["LogisticRegression"].named_steps["model"]),
    ("RF", candidates["RandomForest"].named_steps["model"]),
    ("LGBM", candidates["LightGBM"].named_steps["model"]),
]

candidates["SoftVoting"] = Pipeline(
    steps=[
        ("preprocess", preprocess),
        (
            "model",
            VotingClassifier(
                estimators=estimadores_voting,
                voting="soft",
                n_jobs=1,  # ambiente do wsl estava dando erro de memoria com o parametro -1
            ),
        ),
    ]
)


print("Baseline:", baseline.named_steps["model"])
print("Modelos candidatos:", list(candidates.keys()))


## 7.1 Justificativa dos modelos


- O `DummyClassifier` com `strategy='stratified'` atua como um adversário mais justo, porque utiliza as próprias probabilidades conhecidas das classes para fazer a sua classificação, ao invés de chutar sempre a classe majoritária como um `strategy='most_frequent'` faria.
Ele espelha o comporamento de um sistema baseado em distribuição e frequência histórica dos dados.


- Por serem dados estruturados e tabulares, com milhares de linhas, a escolha de algoritmos tradicionais foi ideal para não aumentar a complexidade desnecessáriamente. A premissa foi alternar entre diferentes estratégias (Linear, Árvore, Bagging, Boosting e Voting) , para ter uma adequada exploração de possibilidades e complexidades.


- LogisticRegression precisa de ajuste de escala, e tanto ela quanto DecisionTree e RandomForest precisam de encoding das variaveis categoricas. LightGBM tem suporte nativo a dados categóricos, mas para padronizar o pipeline o encoding foi de forma global.





# 8. Treinamento e avaliação inicial

In [ ]:
results = {}
trained_models = {}

# Baseline

t0 = time.time()
baseline.fit(X_train, y_train)
train_time = time.time() - t0

y_pred = baseline.predict(X_test)

results["baseline"] = evaluate_classification(y_test, y_pred)
results["baseline"]["train_time_s"] = round(train_time, 3)
trained_models["baseline"] = baseline


for name, model in candidates.items():
    t0 = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - t0

    y_pred = model.predict(X_test)
    results[name] = evaluate_classification(y_test, y_pred)
    results[name]["train_time_s"] = round(train_time, 3)
    trained_models[name] = model


show_results_table(results)


> Nesse resultado preliminar, a métrica principal de F1 Macro foi vencida pelo SoftVoting, mas ele não passou do critério de ao menos 60% de recall na classe `Com vítimas fatais`. Levando isso em consideração o verdadeiro vencedor foi o **LighGBM**.

## 8.1 Análise dos resultados iniciais

- O modelo superou o baseline?
>  Sim o LightGBM superou o baseline em F1-Macro (0.38 x 0.33), e principalmente no recall da classse crítica (Com vítimas fatais), que foi 10x maior que o baseline (0.63, 0.06)

- A métrica escolhida é suficiente para avaliar o problema?
> Não de maneira isolada. Se avaliássemos apenas pelo F1 Macro, o SoftVoting venceria (0.40), mas falharia no critério de sucesso de ter um recall de no mínimo 60% da classe crítica (Com vítimas fatais).

- Algum modelo parece sofrer de underfitting?
> RandomForest e DecisionTree tem um desempenho muito ruim no recall, quase semelhante ao DummyClassifier.

- O tempo de treinamento é aceitável?
> Sim, o Lightgbm com os parametros atuais levou uma média de 6s, bastante viavel e de baixo custo.

- O resultado faz sentido considerando a EDA?
> Embora o F1-Macro geral pareça tímido, esse desempenho é esperado devido ao desbalanceamento severo das classes, e ao fato de utilizarmos estritamente variáveis de contexto restritivas. Para melhorar esse desempenho seriam necessárias outros tipos de variaveis, como velocidade média do trecho no momento, dados de trânsito, dentre outros.


# 9. Validação e otimização de hiperparâmetros

In [ ]:
N_SPLITS = 5
tscv = TimeSeriesSplit(n_splits=N_SPLITS)


N_ITER_SEARCH = 5

model_to_tune = Pipeline(
    steps=[
        ("preprocess", preprocess),
        (
            "model",
            LGBMClassifier(
                objective="multiclass",
                num_class=3,
                class_weight="balanced",
                random_state=SEED,
                verbosity=-1,
                n_jobs=1,
                subsample_freq=1,
            ),
        ),
    ]
)


param_dist = {
    "model__num_leaves": randint(20, 150),
    "model__max_depth": randint(8, 20),
    "model__min_child_samples": randint(25, 500),
    "model__n_estimators": randint(100, 800),

    "model__learning_rate": loguniform(0.01, 0.2),
    "model__reg_alpha": loguniform(0.001, 10),
    "model__reg_lambda": loguniform(0.001, 10),
    "model__min_gain_to_split": loguniform(0.001, 1),

    "model__subsample": uniform(0.6, 0.4),
    "model__colsample_bytree": uniform(0.6, 0.4),
}


scoring = {
    "f1_macro": make_scorer(f1_score, average="macro"),
    "recall_fatal": make_scorer(
        recall_score, labels=["Com vítimas fatais"], average="micro"
    ),
}


search = RandomizedSearchCV(
    model_to_tune,
    param_distributions=param_dist,
    n_iter=N_ITER_SEARCH,
    cv=tscv,
    scoring=scoring,
    refit=False,
    random_state=SEED,
    n_jobs=-1,
    verbose=1,
)

search.fit(X_train, y_train)

results_df = pd.DataFrame(search.cv_results_)

RECALL_MINIMO = 0.60

viaveis = results_df[results_df["mean_test_recall_fatal"] >= RECALL_MINIMO]


if viaveis.empty:
    print(f"Nenhum candidato atingiu o piso de recall de {RECALL_MINIMO}.")
    print(f"Maior recall obtido: {results_df["mean_test_recall_fatal"].max()}")

else:
    melhor = viaveis.loc[viaveis["mean_test_f1_macro"].idxmax()]

    print(f"Candidatos viáveis (recall >= {RECALL_MINIMO}): {len(viaveis)} de {len(results_df)}")
    print(f"F1-Macro: {melhor["mean_test_f1_macro"]}")
    print(f"Recall (Com vítimas fatais): {melhor["mean_test_recall_fatal"]}")
    print(f"Melhores hiperparâmetros: {melhor["params"]}")

    best_model = clone(model_to_tune)
    best_model.set_params(**melhor["params"])
    best_model.fit(X_train, y_train)


## 9.1 Discussão da otimização

Explique o resultado da busca.

**Perguntas para responder:**
- A otimização melhorou o resultado em relação ao modelo inicial?
- A busca foi limitada por tempo, custo ou tamanho da base?
- Os hiperparâmetros escolhidos fazem sentido?
- Você testaria outras combinações se tivesse mais tempo?

**Resposta:**  
> _Preencha aqui._


# 10. Avaliação final no conjunto de teste

Depois de escolher o melhor modelo, avalie-o no conjunto de teste.

**O que incluir:**
- métrica final;
- comparação com baseline;
- análise de erros;
- discussão sobre overfitting/underfitting;
- limitações da solução.

> **Comentário:** a avaliação final deve ser feita em dados não usados para treinar ou escolher hiperparâmetros.


In [ ]:
final_model = best_model
final_mode_name = "LGBM_otimizado"

y_pred = final_model.predict(X_test)

print(classification_report(y_test, y_pred))

plt.figure(figsize=(10, 10))

ConfusionMatrixDisplay.from_estimator(
    final_model, X_test, y_test, cmap="YlGnBu", xticks_rotation="vertical", ax=plt.gca()
)

plt.title("Matriz de Confusão: Modelo Final", fontsize=16)
plt.xlabel("Rótulo Previsto", fontsize=12)
plt.ylabel("Rótulo Real", fontsize=12)
plt.tight_layout()
plt.show()


## 10.1 Análise de erros e limitações

Escreva uma análise crítica dos resultados.

**Perguntas para responder:**
- Quais tipos de erro o modelo comete mais?
- Há sinais de overfitting ou underfitting?
- A métrica escolhida captura bem o objetivo do problema?
- Há viés, limitação de dados ou risco de generalização?
- Em quais cenários o modelo não deveria ser usado?

**Resposta:**  
> _Preencha aqui._


# 11. Comparação final dos modelos

Apresente uma síntese comparativa.

| Modelo | Métrica principal | Outras métricas | Tempo de treino | Observações |
|---|---:|---:|---:|---|
| Baseline | _preencha_ | _preencha_ | _preencha_ | _preencha_ |
| Modelo 1 | _preencha_ | _preencha_ | _preencha_ | _preencha_ |
| Modelo 2 | _preencha_ | _preencha_ | _preencha_ | _preencha_ |
| Modelo otimizado | _preencha_ | _preencha_ | _preencha_ | _preencha_ |

> **Comentário:** esta tabela ajuda o leitor a entender por que o modelo final foi escolhido.


# 12. Boas práticas e rastreabilidade

Documente decisões importantes do projeto.

**O que incluir:**
- seed utilizada;
- principais decisões de pré-processamento;
- modelos testados;
- hiperparâmetros relevantes;
- tempo aproximado de treino;
- recursos computacionais usados;
- limitações conhecidas;
- o que foi tentado e descartado.

**Registro de decisões:**

| Decisão | Justificativa | Impacto esperado |
|---|---|---|
| _ex.: usar F1-score_ | _classes desbalanceadas_ | _avaliar melhor a classe minoritária_ |
| _ex.: remover coluna X_ | _vazamento de dados_ | _evitar desempenho artificial_ |
| _ex.: usar Random Forest_ | _capturar não linearidades_ | _melhorar baseline_ |


# 13. Conclusão

Faça o fechamento do MVP conectando o resultado ao problema inicial.

**O que incluir:**
- objetivo do trabalho;
- melhor solução encontrada;
- comparação com baseline;
- principais aprendizados;
- limitações;
- próximos passos.

**Conclusão:**  
> _Preencha aqui._

> **Comentário:** uma boa conclusão não repete apenas métricas. Ela explica o que os resultados significam no contexto do problema.


# 14. Salvamento de artefatos

Esta seção é opcional, mas recomendada quando o treinamento for demorado.

**O que pode ser salvo:**
- pipeline final;
- modelo treinado;
- encoder/scaler;
- tabela de resultados;
- gráficos importantes.

> **Comentário:** se salvar arquivos, garanta que o notebook continue executando do início ao fim. Não dependa de arquivos locais que o professor não terá.


# 15. Apêndice opcional: Deep Learning, Fine-tuning ou métodos avançados

Use esta seção apenas se o projeto realmente precisar.

**O que documentar se usar deep learning/fine-tuning:**
- arquitetura ou modelo pré-treinado;
- preparação específica dos dados;
- tamanho de batch;
- número de épocas;
- função de perda;
- otimizador;
- early stopping;
- uso de GPU/CPU;
- tempo de treino;
- comparação com baseline simples.

> **Comentário:** deep learning não é obrigatório. Um modelo clássico bem avaliado pode ser uma solução melhor para muitos MVPs.
